In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
from scipy.stats import trim_mean
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
import default_risk.config as cfg
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
import dtale

installments_payment_df = pd.read_parquet(cfg.CLEANS_DIR / "installments_payments.train-cleaned.parquet")

column_order_reference="days_instalment"

installments_payment_df.sort_values(["id_prev",column_order_reference,"days_instalment"],inplace=True)

data_frame_size=len(installments_payment_df)

installments_payment_df.head()


,id_prev,id_curr,num_instalment_version,num_instalment_number,days_instalment,days_entry_payment,days_entry_payment_is_missing,amt_instalment,amt_payment
435948,1000001,158271,1.0,1,-268.0,-294.0,0,6404.310,6404.310
1836232,1000001,158271,2.0,2,-238.0,-244.0,0,62039.115,62039.115
5232437,1000003,252457,1.0,1,-94.0,-108.0,0,4951.350,4951.350
6007398,1000003,252457,1.0,2,-64.0,-81.0,0,4951.350,4951.350
3640493,1000003,252457,1.0,3,-34.0,-49.0,0,4951.350,4951.350


In [ ]:
installments_payment_df["intial_version"]= installments_payment_df.groupby("id_prev")["num_instalment_version"].transform("first")


In [ ]:
#we starting catching this because we are probably cutting some parts of the temporal sequence
installments_payment_df["raw_size_serie"]= installments_payment_df.groupby("id_prev").transform("size")
installments_payment_df["ammount_of_versions_in_sequence"] = installments_payment_df["num_instalment_version"].transform("nunique")

In [ ]:
#we gonna use the knowlege recolected in the EDA. 
#For more details look eda_installments_payments.ipynb decisions summary 1#

#creating auxiliar columns
installments_payment_df["next_payment_value"] = installments_payment_df.groupby("id_prev")["days_entry_payment"].shift(-1)
nan_amount= installments_payment_df.groupby("id_prev")["days_entry_payment_is_missing"].transform("sum")

#defining the mask to separate the differents cases of missings values 
have_missing_entry_payment_mask= (installments_payment_df["next_payment_value"].isna())
not_a_deadtail_mask= (installments_payment_df["days_entry_payment"].isna() ) & ( installments_payment_df["next_payment_value"].notna())

#we want to catch the cases of  dead-tail so we starting filtering that cases with nans but that are not dead tails
non_dead_tail_nans= installments_payment_df[not_a_deadtail_mask]
ids_with_nulls_that_are_non_deadtails= non_dead_tail_nans["id_prev"].unique()
excluding_nans_non_deadtails_mask= ~(installments_payment_df["id_prev"].isin(ids_with_nulls_that_are_non_deadtails))
series_without_problematic_nans = installments_payment_df[excluding_nans_non_deadtails_mask]

#now this ID are series where the nans are deadtails, so count nans in "days_entry_payment" o "amt_payment" is calculate
#the lenght of the deadtail and we save that value in a new column
ids_with_deadtails= series_without_problematic_nans[series_without_problematic_nans["days_entry_payment"].isna()]["id_prev"].unique()
installments_payment_df["dead_tail_lenght"]= np.where(installments_payment_df["id_prev"].isin(ids_with_deadtails),  nan_amount, 0)

In [7]:
dtale.show(installments_payment_df [installments_payment_df["dead_tail_lenght"] !=0 ])